### Create tools:

In [1]:
from langchain_core.tools import tool

class VehicleClimate:
    def __init__(self, initial_temperature: float = 21.5):
        self._interior_temperature = initial_temperature

    def get_vehicle_interior_temperature(self) -> str:
        """Returns the current vehicle interior temperature."""
        print(">>> Executing 'get_vehicle_interior_temperature' <<<\n")
        return f"The current vehicle interior temperature is {self._interior_temperature:.1f}°C"

    def set_vehicle_interior_temperature(self, temperature: float) -> str:
        """Sets the vehicle interior temperature to the requested value."""
        print(f"\n>>> Executing 'set_vehicle_interior_temperature' to {temperature}°C <<<\n")
        self._interior_temperature = temperature
        return f"Vehicle interior temperature has been set to {self._interior_temperature:.1f}°C"


vehicle_climate = VehicleClimate()

@tool
def get_vehicle_interior_temperature() -> str:
    """Reads the current vehicle interior temperature."""
    return vehicle_climate.get_vehicle_interior_temperature()


@tool
def set_vehicle_interior_temperature(temperature: float) -> str:
    """Sets the vehicle interior temperature to a new value in degrees Celsius."""
    return vehicle_climate.set_vehicle_interior_temperature(temperature)

### Load model and create agent:

In [6]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent

llm = ChatOllama(model='qwen3', temperature=0)
# llm = ChatOllama(model='qwen3', temperature=0, base_url="http://127.0.0.1:11434",)

agent = create_agent(
    model=llm,
    tools=[get_vehicle_interior_temperature, set_vehicle_interior_temperature],
    system_prompt="You are a helpful assistant",
)

### Call model:

In [7]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Set the vehicle interior temperature to 24 degrees and then tell me the current value.",
            }
        ]
    }
)


>>> Executing 'set_vehicle_interior_temperature' to 24.0°C <<<

>>> Executing 'get_vehicle_interior_temperature' <<<



### Print full conversation:

In [8]:
print("** Full conversation: **")
for m in result["messages"]:
    print(f" {type(m).__name__}:\n  content={m.content}\n  tool_calls={m.tool_calls if hasattr(m, 'tool_calls') else 'N/A'}\n")
    
print("\n** Final AI response: **")
print(result["messages"][-1].content)

** Full conversation: **
 HumanMessage:
  content=Set the vehicle interior temperature to 24 degrees and then tell me the current value.
  tool_calls=N/A

 AIMessage:
  content=
  tool_calls=[{'name': 'set_vehicle_interior_temperature', 'args': {'temperature': 24}, 'id': '2f0ff8d0-57c3-46ef-a3ab-0139fb9e0bf6', 'type': 'tool_call'}, {'name': 'get_vehicle_interior_temperature', 'args': {}, 'id': '8c19d5d0-eb96-459b-a8ad-fa324bf21c0f', 'type': 'tool_call'}]

 ToolMessage:
  content=Vehicle interior temperature has been set to 24.0°C
  tool_calls=N/A

 ToolMessage:
  content=The current vehicle interior temperature is 24.0°C
  tool_calls=N/A

 AIMessage:
  content=The vehicle interior temperature has been successfully set to **24.0°C**, and the current value is **24.0°C**.
  tool_calls=[]


** Final AI response: **
The vehicle interior temperature has been successfully set to **24.0°C**, and the current value is **24.0°C**.
